In [ ]:
from pathlib import Path
import os

TASK_PATH = Path("simulate_python/task_1_standing")
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / TASK_PATH).is_dir())
TASK_DIR = REPO_ROOT / TASK_PATH
os.chdir(TASK_DIR)
print("Working directory:", TASK_DIR)


In [ ]:
import numpy as np
import os
import pandas as pd
import tensorflow as tf
from tensorflow import keras
import time
WINDOW_SIZE = 10

In [ ]:
X = []
Y = []
for i in range(2000):
    df = pd.read_csv('./csv/'+str(i)+'.csv')
    df = df.iloc[:210,:]
    df['acc_mag'] = np.sqrt(df['acc_x']**2 + df['acc_y']**2 + df['acc_z']**2)
    df['gyro_mag'] = np.sqrt(df['gyro_x']**2 + df['gyro_y']**2 + df['gyro_z']**2)
    for i in range(200):
        data = df.iloc[i:i+WINDOW_SIZE, -3:]
        X.append(data)
        if df.iloc[i,0] > 0.2:
            Y.append(1)
        else:
            Y.append(0)
X = np.array(X)
Y = np.array(Y)

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.LSTM(16, use_bias=False, return_sequences=False,
                         input_shape=(10,3)),
    tf.keras.layers.Dense(2, activation='softmax', use_bias=False)
])
model.compile('adam','sparse_categorical_crossentropy',metrics=['acc'])
model.fit(X,Y,batch_size=256,epochs=10,verbose=0)

model.save("0210.keras")

In [ ]:
test_df = pd.read_csv('./csv/1237.csv')
test_df = test_df.iloc[:210,:]
test_df['acc_mag'] = np.sqrt(test_df['acc_x']**2 + test_df['acc_y']**2 + test_df['acc_z']**2)
test_df['gyro_mag'] = np.sqrt(test_df['gyro_x']**2 + test_df['gyro_y']**2 + test_df['gyro_z']**2)
predictions = []
input_x = []
model = keras.models.load_model('0210.keras')
# for i in range(200):
#     data = test_df.iloc[i:i+WINDOW_SIZE, -3:]
#     input_x.append(data)
# input_x = np.array(input_x)    
# probs = model.predict(input_x)
# pred  = probs.argmax(axis=1)
# print(pred) 

for i in range(10):
    data = test_df.iloc[i:i+WINDOW_SIZE, -3:]
    data = np.array(data)
    data = np.expand_dims(data, axis=0) 
    start = time.time()
    probs = model.predict(data)
    end = time.time()
    pred  = probs.argmax(axis=1)
    print(pred) 
    print(f"Prediction Duration: {end - start}")